In [1]:
# Cell 1: check GPU, clone repo
!nvidia-smi | head -15

import os
REPO = "/kaggle/working/ronlp-project"
if not os.path.exists(f"{REPO}/src"):
    if os.path.exists(REPO):
        !rm -rf {REPO}
    !git clone https://github.com/Sruf-y/ronlp-project.git {REPO}
else:
    !cd {REPO} && git pull

%cd {REPO}
print("cwd:", os.getcwd())
print("src exists:", os.path.exists(f"{REPO}/src"))

Sat Nov 21 18:41:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cell 2: install (Kaggle Python 3.11 — numpy<2 works natively with datasets 2.18)
!pip uninstall -y datasets
!rm -rf /usr/local/lib/python3.13/dist-packages/datasets
!rm -rf /usr/local/lib/python3.13/dist-packages/datasets-*.dist-info
!pip install "datasets==2.18.0" --no-cache-dir

import numpy, datasets, transformers, torch
print("numpy:", numpy.__version__)          # 1.26.x
print("datasets:", datasets.__version__)    # 2.18.0
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [fsspec]
    Found existing installation: dill 0.4.1━━━━━━━━━━━━━━━━━━━ 1/5 [fsspec]
    Uninstalling dill-0.4.1:90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [fsspec]
      Successfully uninstalled dill-0.4.10m━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [dill]
  Attempting uninstall: multiprocess90m━━━━━━━━━━━━━━━━━━━━━━━ 2/5 [dill]
    Found existing installation: multiprocess 0.70.19━━━━━━━━━ 2/5 [dill]
    Uninstalling multiprocess-0.70.19:90m╺━━━━━━━━━━━━━━━ 3/5 [multiprocess]
      Successfully uninstalled multiproces

In [6]:
# Cell 2.5: Check the max length. Use 128 if the 95th percentile is under 256. Use 256 otherwise
lengths = [len(tok) for tok in tokenizer(X_train)["input_ids"]]
import numpy as np
print("median:", np.median(lengths))
print("95th percentile:", np.percentile(lengths, 95))
print("max:", max(lengths))

NameError: name 'tokenizer' is not defined

In [7]:
# Cell 3: load + tokenize + wrap in plain PyTorch Dataset (avoids datasets bug entirely)
import sys, os, importlib.util
import torch
from torch.utils.data import Dataset as TorchDataset

REPO = "/kaggle/working/ronlp-project"

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

data_utils = load_module("data_utils", f"{REPO}/src/data_utils.py")
preprocess = load_module("preprocess", f"{REPO}/src/preprocess.py")

get_laroseda = data_utils.get_laroseda
to_binary = preprocess.to_binary
combine_text = preprocess.combine_text
print("✅ imports OK")

laroseda = get_laroseda()
print(laroseda)

def to_examples(ds):
    X = [combine_text(t, c) for t, c in zip(ds["title"], ds["content"])]
    y = [to_binary(r) for r in ds["starRating"]]
    return X, y

X_train, y_train = to_examples(laroseda["train"])
X_test,  y_test  = to_examples(laroseda["test"])
print(f"train: {len(X_train)}, test: {len(X_test)}")

# Tokenize as plain Python lists
from transformers import AutoTokenizer
MODEL_NAME = "dumitrescustefan/bert-base-romanian-cased-v1"
MAX_LEN = 128 # use 128 if 95th percentile from before was under 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

enc_train = tokenizer(X_train, truncation=True, padding="max_length", max_length=MAX_LEN)
enc_test  = tokenizer(X_test,  truncation=True, padding="max_length", max_length=MAX_LEN)

# Plain PyTorch Dataset — no `datasets` lib involvement in batching
class SentimentDataset(TorchDataset):
    def __init__(self, enc, labels):
        self.input_ids      = torch.tensor(enc["input_ids"], dtype=torch.long)
        self.attention_mask = torch.tensor(enc["attention_mask"], dtype=torch.long)
        self.labels         = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

train_tok = SentimentDataset(enc_train, y_train)
test_tok  = SentimentDataset(enc_test,  y_test)
print(f"train_tok: {len(train_tok)}, test_tok: {len(test_tok)}")
print("sample:", {k: tuple(v.shape) for k, v in train_tok[0].items()})

✅ imports OK


/usr/local/lib/python3.13/dist-packages/datasets/load.py:1461: FutureWarning: The repository for universityofbucharest/laroseda contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/universityofbucharest/laroseda
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/12000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['index', 'title', 'content', 'starRating'],
        num_rows: 12000
    })
    test: Dataset({
        features: ['index', 'title', 'content', 'starRating'],
        num_rows: 3000
    })
})
train: 12000, test: 3000


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/397k [00:00<?, ?B/s]

train_tok: 12000, test_tok: 3000
sample: {'input_ids': (128,), 'attention_mask': (128,), 'labels': ()}


In [8]:
# Cell 4: metrics
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

In [9]:
# Cell 5: model + TrainingArguments
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1},
)

OUTPUT_DIR = "/kaggle/working/bert-ro-sentiment"

args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=64, #was 16, was too slow
    per_device_eval_batch_size=128, #was 32, same reason it was too slow
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    seed=42,
    report_to="none",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
print("Trainer ready")

model.safetensors: reconstructing file:   0%|          |  0.00B /  500MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

Trainer ready


In [10]:
# Cell 6: train (this is the one that takes ~15-20 min on T4)
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.119528,0.982667,0.982666
2,0.233818,0.118511,0.983333,0.983333
3,0.062420,0.125983,0.984333,0.984333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=282, training_loss=0.11805925217080623, metrics={'train_runtime': 465.068, 'train_samples_per_second': 77.408, 'train_steps_per_second': 0.606, 'total_flos': 2367999498240000.0, 'train_loss': 0.11805925217080623, 'epoch': 3.0})

In [11]:
# Cell 7: evaluate
results = trainer.evaluate()
print(f"accuracy:  {results['eval_accuracy']:.4f}")
print(f"macro-F1:  {results['eval_macro_f1']:.4f}")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.062420,0.125983,3,0.984333,0.984333


accuracy:  0.9843
macro-F1:  0.9843


In [12]:
# Cell 8: save — ends up in Kaggle Output tab
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("saved to", OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to /kaggle/working/bert-ro-sentiment


In [13]:
# Cell 9: custom predictions (compare with your TF-IDF results)
from transformers import pipeline

sentiment = pipeline(
    "text-classification",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
)

custom = [
    "Foarte mulțumit. Produsul a sosit repede și funcționează perfect. Recomand!",
    "Dezastru. Nu funcționează deloc, banii aruncați. Nu recomand nimănui.",
    "Okay. Nu e nici bun nici rău, își face treaba.",
    "Baterie bună. Ține toată ziua, dar camera e slabă.",
    "Perfect! Nu am avut nici o problema la utilizarea produsului!",
]

for text in custom:
    r = sentiment(text)[0]
    print(f"{r['label']:8s} ({r['score']:.3f})  |  {text[:70]}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

POSITIVE (0.999)  |  Foarte mulțumit. Produsul a sosit repede și funcționează perfect. Reco
NEGATIVE (0.998)  |  Dezastru. Nu funcționează deloc, banii aruncați. Nu recomand nimănui.
POSITIVE (0.970)  |  Okay. Nu e nici bun nici rău, își face treaba.
POSITIVE (0.929)  |  Baterie bună. Ține toată ziua, dar camera e slabă.
POSITIVE (0.996)  |  Perfect! Nu am avut nici o problema la utilizarea produsului!


| Model | Accuracy | Macro-F1 |
|---|---|---|
| Majority class | 0.5000 | 0.3333 |
| TF-IDF + LogisticRegression | 0.9613 | 0.9613 |
| **BERT-ro (dumitrescustefan/bert-base-romanian-cased-v1)** | **0.9843** | **0.9843** |

| Input | TF-IDF | BERT | DIFFERENCE |
|---|---|---|---|
| "Foarte mulțumit. Produsul a sosit repede și funcționează perfect. Recomand!" | POSITIVE (0.985) | POSITIVE (0.999) |  +0.014|
| "Dezastru. Nu funcționează deloc, banii aruncați. Nu recomand nimănui." | NEGATIVE (0.965) | NEGATIVE (0.998) | +0.033 |
| "Okay. Nu e nici bun nici rău, își face treaba." | POSITIVE (0.665) | POSITIVE (0.970) | +0.305 |
| "Baterie bună. Ține toată ziua, dar camera e slabă." | POSITIVE (0.552) | POSITIVE (0.929) | +0.377 |
|Perfect! Nu am avut nici o problema la utilizarea produsului!| POSITIVE (0.649) |POSITIVE (0.996)| +0.347 |